# YouTube Comment Sentiment Analysis — DistilBERT Fine-Tuning
**3-class classifier (Positive / Neutral / Negative) · Full fine-tuning · Google Colab + Drive**

## Step 1 — Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q datasets transformers accelerate evaluate scikit-learn pandas matplotlib seaborn

In [ ]:
import os, re, json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_dataset, DatasetDict, load_from_disk
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)
import evaluate

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# GPU check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Project paths (Google Drive)
PROJECT_PATH = "/content/drive/MyDrive/RESUME-PROJECTS/NLP/YT-Sentiment-Analysis-Project/"
os.makedirs(PROJECT_PATH, exist_ok=True)

RAW_DATA = PROJECT_PATH + "raw_data/"
PROCESSED_DATA = PROJECT_PATH + "processed_data/"
MODEL_CHECKPOINTS = PROJECT_PATH + "model_checkpoints/"
FINAL_MODEL = PROJECT_PATH + "final_model/"
OUTPUTS = PROJECT_PATH + "outputs/"

for p in [RAW_DATA, PROCESSED_DATA, MODEL_CHECKPOINTS, FINAL_MODEL, OUTPUTS]:
    os.makedirs(p, exist_ok=True)
print("Project directories ready.")

## Step 2 — Data Import

In [ ]:
raw_dataset = load_dataset("AmaanP314/youtube-comment-sentiment")
print(raw_dataset)
print(raw_dataset["train"][0])

In [ ]:
# Cache raw data to Drive for resumability
raw_dataset.save_to_disk(RAW_DATA)
print(f"Raw data saved to {RAW_DATA}")

## Step 3 — Data Cleaning

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert the raw_dataset's train split to a pandas DataFrame
df_country_code = raw_dataset["train"].to_pandas()

# Visualize data size by 'CountryCode'
country_counts = df_country_code['CountryCode'].value_counts().nlargest(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=country_counts.index, y=country_counts.values, palette='viridis')
plt.title('Top 10 Country Codes by Comment Count')
plt.xlabel('Country Code')
plt.ylabel('Number of Comments')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Top 10 Country Codes by Comment Count:")
print(country_counts)


In [ ]:
# Keep only English Comments
english_speaking_countries = ['US', 'AU', 'GB', 'NZ']
eng_dataset = raw_dataset.filter(lambda example: example['CountryCode'] in english_speaking_countries)


In [ ]:
df = eng_dataset["train"].to_pandas()
initial_len = len(df)
print(f"Initial rows: {initial_len:,}")

# Keep only needed columns
df = df[["CommentText", "Sentiment"]].copy()

# Remove duplicates
df.drop_duplicates(subset="CommentText", keep="first", inplace=True)
print(f"After dedup: {len(df):,} (removed {initial_len - len(df):,})")

# Remove URLs and links
url_pattern = r'\b(?:https?|ftp|file|mailto):\/\/\S+|(?:www\.)\S+\.\S+\b'
df["CommentText"] = df["CommentText"].astype(str).apply(lambda x: re.sub(url_pattern, '', x))

# Remove timestamps (e.g., 2:30, 01:23:45)
df["CommentText"] = df["CommentText"].apply(lambda x: re.sub(r'\b\d{1,2}:\d{2}(?::\d{2})?\b', '', x))

# Normalize whitespace
df["CommentText"] = df["CommentText"].str.strip().apply(lambda x: re.sub(r'\s+', ' ', x))

# Remove empty or very short comments (< 3 chars)
df = df[df["CommentText"].str.len() >= 3].reset_index(drop=True)
print(f"After cleaning: {len(df):,} (total removed: {initial_len - len(df):,})")
print(f"\nSentiment distribution:\n{df['Sentiment'].value_counts()}")

# Save cleaned dataframe as CSV
df.to_csv(OUTPUTS + "cleaned_comments.csv", index=False)
print(f"Cleaned data saved → {OUTPUTS}cleaned_comments.csv  ({len(df):,} rows)")

## Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
# Load cleaned dataframe from CSV
df_cleaned = pd.read_csv(OUTPUTS + "cleaned_comments.csv")
print(f"Cleaned data loaded from {OUTPUTS}cleaned_comments.csv ({len(df_cleaned):,} rows)")
print("\nFirst 5 rows of the loaded 'df_cleaned':")
print(df_cleaned.head())

# Create a copy for EDA to preserve the original cleaned state
df = df_cleaned.copy()
print(f"'df' created with {len(df):,} rows for further analysis.")
print("\nFirst 5 rows of the loaded 'df':")
print(df.head())


In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
MAX_LENGTH = 64

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(7, 4))
counts = df["Sentiment"].value_counts()
colors = {"Positive": "#2ecc71", "Neutral": "#3498db", "Negative": "#e74c3c"}
counts.plot(kind="bar", color=[colors[c] for c in counts.index], ax=ax, edgecolor="black")
for i, (v, pct) in enumerate(zip(counts.values, counts.values / len(df) * 100)):
    ax.text(i, v + len(df)*0.005, f"{v:,}\n({pct:.1f}%)", ha="center", fontsize=9)
ax.set_title("Class Distribution")
ax.set_ylabel("Count")
ax.set_xticklabels(counts.index, rotation=0)
plt.tight_layout()
plt.savefig(OUTPUTS + "class_distribution.png", dpi=150)
plt.show()

In [ ]:
# Comment length distribution (characters)
df["char_len"] = df["CommentText"].str.len()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df["char_len"].hist(bins=100, ax=axes[0], color="#3498db", edgecolor="black")
axes[0].set_title("Character Length Distribution")
axes[0].set_xlabel("Characters")
axes[0].axvline(df["char_len"].median(), color="red", linestyle="--", label=f'Median: {df["char_len"].median():.0f}')
axes[0].legend()

# Token count distribution (quick estimate using whitespace split)
# df["word_count"] = df["CommentText"].str.split().str.len()
# df["word_count"].hist(bins=100, ax=axes[1], color="#2ecc71", edgecolor="black")
# axes[1].set_title("Word Count Distribution")
# axes[1].set_xlabel("Words")
df["token_count"] = df["CommentText"].apply(lambda x: len(tokenizer.tokenize(x)))
df["token_count"].hist(bins=100, ax=axes[1], color="#2ecc71", edgecolor="black")
axes[1].set_title("Token Count Distribution (DistilBERT)")
axes[1].set_xlabel("Tokens")
axes[1].axvline(64, color="red", linestyle="--", label="max_length=64 tokens")
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUTS + "length_distribution.png", dpi=150)
plt.show()

# stats = df[["char_len", "word_count"]].describe().T
stats = df[["char_len", "token_count"]].describe().T
stats.to_csv(OUTPUTS + "dataset_stats.csv")
print(stats)
# df.drop(columns=["char_len", "word_count"], inplace=True)
# df.drop(columns=["char_len", "token_count"], inplace=True)


In [ ]:
# Sample comments per class
samples = df.groupby("Sentiment").apply(lambda x: x.sample(5, random_state=SEED)).reset_index(drop=True)
samples.to_csv(OUTPUTS + "sample_comments.csv", index=False)
for sentiment in ["Positive", "Neutral", "Negative"]:
    print(f"\n--- {sentiment} ---")
    for _, row in samples[samples["Sentiment"] == sentiment].iterrows():
        print(f"  {row['CommentText'][:120]}")

In [ ]:
# Save EDA dataframe as CSV
df.to_csv(OUTPUTS + "eda_comments.csv", index=False)
print(f"EDA data saved → {OUTPUTS}eda_comments.csv  ({len(df):,} rows)")

## Step 5 — Data Preprocessing & Tokenization

In [ ]:
# Load cleaned dataframe from CSV
df_eda = pd.read_csv(OUTPUTS + "eda_comments.csv")
print(f"EDA data loaded from {OUTPUTS}eda_comments.csv ({len(df_eda):,} rows)")
print("\nFirst 5 rows of the loaded 'df_eda':")
print(df_eda.head())

# Create a copy for EDA to preserve the original cleaned state
df = df_eda.copy()
print(f"'df' created with {len(df):,} rows for further analysis.")
print("\nFirst 5 rows of the loaded 'df':")
print(df.head())


In [ ]:
print(df.columns.tolist())

# Label encoding: Positive→0, Neutral→1, Negative→2
LABEL_MAP = {"Positive": 0, "Neutral": 1, "Negative": 2}
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}

df["label"] = df["Sentiment"].map(LABEL_MAP)
df.rename(columns={"CommentText": "text"}, inplace=True)
df_standby = df
df = df[["text", "label", "token_count"]].copy()

print(f"Label distribution:\n{df['label'].value_counts().sort_index()}")
print(f"\nTotal samples: {len(df):,}")

In [ ]:
print(f"Total samples: {len(df):,}")
before = len(df)
print(f"Total before : {before}")

In [ ]:
# Filter out comments exceeding MAX_LENGTH tokens
# before = len(df)
df = df[df["token_count"] <= MAX_LENGTH].reset_index(drop=True)
print(f"After token filter (≤{MAX_LENGTH}): {len(df):,} (removed {before - len(df):,})")
# df.drop(columns=["char_len", "token_count"], inplace=True)
df.drop(columns=["token_count"], inplace=True)

In [ ]:
print(f"\nLabel distribution after filtering:\n{df['label'].value_counts().sort_index()}")
print(f"\nTotal samples: {len(df):,}")

In [ ]:
# Stratified sampling: 50,000 samples for each sentiment label
sampled_dfs = []
for label_val in df['label'].unique():
    n_samples = min(150000, len(df[df['label'] == label_val]))
    sampled_dfs.append(df[df['label'] == label_val].sample(n=n_samples, random_state=SEED))

df_chosen = pd.concat(sampled_dfs).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Created 'df_chosen' with {len(df_chosen):,} rows after stratified sampling.")
print(f"Label distribution in 'df_chosen':\n{df_chosen['label'].value_counts().sort_index()}")

# Save the chosen dataset to CSV
df_chosen.to_csv(OUTPUTS + "chosen_dataset_150000.csv", index=False)
print(f"Chosen dataset saved → {OUTPUTS}chosen_dataset_150000.csv  ({len(df_chosen):,} rows)")


In [ ]:
# Load cleaned dataframe from CSV
df_chosen = pd.read_csv(OUTPUTS + "chosen_dataset_150000.csv")
print(f"'df_chosen' created with {len(df_chosen):,} rows for further analysis.")
print(df_chosen.head())

# Create a copy to preserve the original state
df = df_chosen.copy()
print(f"'df' updated with {len(df):,} rows for subsequent splitting.")
print(df.head())


In [ ]:
# Stratified split: 80% train / 10% val / 10% test
from datasets import Dataset
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["label"])

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

print(f"Train: {len(dataset['train']):,}  |  Val: {len(dataset['validation']):,}  |  Test: {len(dataset['test']):,}")

In [ ]:
# Tokenize

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

tokenized_dataset = dataset.map(tokenize_fn, batched=True, batch_size=1000)
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Tokenization complete.")
print(tokenized_dataset)

In [ ]:
# Save tokenized dataset to Drive
tokenized_dataset.save_to_disk(PROCESSED_DATA)
print(f"Processed data saved to {PROCESSED_DATA}")

## Step 6 — Model Import

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q evaluate

In [3]:
import os
import json
import torch
import evaluate
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import (
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import load_from_disk


In [4]:
# Define device for model placement
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Label encoding: Positive→0, Neutral→1, Negative→2
LABEL_MAP = {"Positive": 0, "Neutral": 1, "Negative": 2}
ID2LABEL = {v: k for k, v in LABEL_MAP.items()}


In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL_MAP,
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,}  |  Trainable: {trainable_params:,}  |  All trainable: {total_params == trainable_params}")

## Step 7 — Fine-Tuning

In [6]:
# Project paths (Google Drive)
PROJECT_PATH = "/content/drive/MyDrive/RESUME-PROJECTS/NLP/YT-Sentiment-Analysis-Project/"
RAW_DATA = PROJECT_PATH + "raw_data/"
PROCESSED_DATA = PROJECT_PATH + "processed_data/"
MODEL_CHECKPOINTS = PROJECT_PATH + "model_checkpoints/"
FINAL_MODEL = PROJECT_PATH + "final_model/"
OUTPUTS = PROJECT_PATH + "outputs/"
SEED = 42


In [ ]:
# Load tokenized dataset from Drive
tokenized_dataset = load_from_disk(PROCESSED_DATA)
print(f"Tokenized dataset loaded from {PROCESSED_DATA}")
print(tokenized_dataset)


In [ ]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1}

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_CHECKPOINTS,
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=True,
    label_smoothing_factor=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Training configuration ready. Starting training...")

In [12]:
# Resume from checkpoint if available, otherwise start fresh
import glob
checkpoints = glob.glob(os.path.join(MODEL_CHECKPOINTS, "checkpoint-*"))
resume_ckpt = max(checkpoints, key=os.path.getmtime) if checkpoints else None
if resume_ckpt:
    print(f"Resuming from: {resume_ckpt}")


In [ ]:
# Train
trainer.train(resume_from_checkpoint=resume_ckpt)

## Step 8 — Model Evaluation

In [15]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

In [ ]:
# Predict on test set
predictions = trainer.predict(tokenized_dataset["test"])
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Metrics
acc = accuracy_score(labels, preds)
macro_f1 = f1_score(labels, preds, average="macro")
macro_prec = precision_score(labels, preds, average="macro")
macro_rec = recall_score(labels, preds, average="macro")

metrics = {
    "accuracy": round(acc, 4),
    "macro_f1": round(macro_f1, 4),
    "macro_precision": round(macro_prec, 4),
    "macro_recall": round(macro_rec, 4),
}
print("=== Test Set Results ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

with open(OUTPUTS + "test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

In [ ]:
# Classification report
label_names = ["Positive", "Neutral", "Negative"]
report = classification_report(labels, preds, target_names=label_names, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(OUTPUTS + "classification_report.csv")
print(classification_report(labels, preds, target_names=label_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix  |  Acc: {acc:.4f}  |  F1: {macro_f1:.4f}")
plt.tight_layout()
plt.savefig(OUTPUTS + "confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# Training loss & eval metric curves
log_history = trainer.state.log_history
train_logs = [x for x in log_history if "loss" in x and "eval_loss" not in x]
eval_logs = [x for x in log_history if "eval_f1" in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if train_logs:
    axes[0].plot([x["step"] for x in train_logs], [x["loss"] for x in train_logs], color="#e74c3c")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")

if eval_logs:
    axes[1].plot([x["step"] for x in eval_logs], [x["eval_accuracy"] for x in eval_logs], label="Accuracy", marker="o")
    axes[1].plot([x["step"] for x in eval_logs], [x["eval_f1"] for x in eval_logs], label="Macro F1", marker="s")
    axes[1].set_title("Eval Metrics Over Time")
    axes[1].set_xlabel("Step")
    axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUTS + "training_curves.png", dpi=150)
plt.show()

## Step 9 — Save Final Model

In [ ]:
import os
from transformers import DistilBertTokenizer

# Ensure tokenizer is defined. This handles cases where earlier cells might not have run (e.g., kernel restart).
if 'tokenizer' not in globals() or not isinstance(tokenizer, DistilBertTokenizer):
    tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

model.save_pretrained(FINAL_MODEL)
tokenizer.save_pretrained(FINAL_MODEL)

saved_files = os.listdir(FINAL_MODEL)
model_size_mb = sum(os.path.getsize(os.path.join(FINAL_MODEL, f)) for f in saved_files) / 1e6
print(f"Model saved to {FINAL_MODEL}")
print(f"Files: {saved_files}")
print(f"Total size: {model_size_mb:.1f} MB")

## Step 10 — Inference

In [30]:
MAX_LENGTH = 64

In [ ]:
# Load model from final_model/ (proves it's reloadable)
inf_tokenizer = DistilBertTokenizer.from_pretrained(FINAL_MODEL)
inf_model = DistilBertForSequenceClassification.from_pretrained(FINAL_MODEL)
inf_model.to(device)
inf_model.eval()

def predict(comment: str):
    inputs = inf_tokenizer(comment, return_tensors="pt", padding="max_length", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = inf_model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    pred_id = torch.argmax(probs, dim=-1).item()
    return ID2LABEL[pred_id], round(probs[0][pred_id].item(), 4)

In [ ]:
test_comments = [
    "This is the best video I've ever watched! So helpful!",
    "Absolutely terrible, waste of my time.",
    "The video was okay, nothing special.",
    "Love your content, keep it up! 🔥",
    "I don't understand why people like this.",
    "Interesting perspective, but I disagree with some points.",
    "First",
    "Can someone explain what he said at the beginning?",
    "This changed my life, thank you so much!!!",
    "Boring. Disliked.",
    "Not bad, could be better though.",
    "Who else is watching this in 2026? 😂",
    "The editing is great but the info is wrong.",
    "Subscribed! Best channel for this topic.",
    "meh",
    "I tried this and it actually works perfectly",
    "Stop clickbaiting, the title is misleading",
    "Fair review, balanced and honest",
    "LMAOOO this is hilarious 😂😂😂",
    "Didn't learn anything new unfortunately",
]

results = []
for c in test_comments:
    label, conf = predict(c)
    results.append({"comment": c, "predicted_label": label, "confidence": conf})

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUTS + "inference_results.csv", index=False)

for _, row in results_df.iterrows():
    print(f"[{row['predicted_label']:>8}] ({row['confidence']:.2f})  {row['comment'][:80]}")

In [ ]:
print("\n=== All outputs saved to:", OUTPUTS, "===")
print("Files:", os.listdir(OUTPUTS))
print("\nDone! ✅")

# ALL DONE